# CMIP6 Model Validation — Single Best-Model Identification
## Reference Period: 1995–2014

**Purpose:**  
Evaluate the performance of six CMIP6 models in reproducing observed monthly precipitation climatology at 49 station locations across Jordan's hydrological basins. Model performance is assessed using five complementary statistical metrics, and a composite ranking system identifies the best-performing model for each basin.

**Models evaluated:**  
CMCC-CM2-SR5 · CNRM-ESM2-1 · EC-Earth3-Veg · IPSL-CM6A-LR · MPI-ESM1-2-LR · NorESM2-MM

**Evaluation approach** (Hussien et al.):
1. Compute five performance metrics at each station for each model  
2. Aggregate station-level metrics to basin level (mean across stations in basin)  
3. Rank models within each basin using a composite ranking system  
4. Identify the single best-performing model per basin (lowest average rank)  
5. Construct a 3-model ensemble (top 3) and 6-model ensemble for comparison

**Outputs saved to:**  
`C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\single.model\`




## 1. Import Libraries


In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial import cKDTree
from itertools import combinations

warnings.filterwarnings("ignore")

print("Libraries loaded.")
print(f"  numpy  : {np.__version__}")
print(f"  pandas : {pd.__version__}")


Libraries loaded.
  numpy  : 2.1.3
  pandas : 2.2.3


## 2. Configuration


In [2]:
# ── Input paths ──────────────────────────────────────────────────────────────
# Station mapping (49 stations, basin assignments)
STATION_MAPPING_FILE = (
    r"D:\RICAAR\Pr.New.Stations.Selection\OBSERVATIONS"
    r"\Station_Basin_Mapping.xlsx"
)

# Observed LTMM (from Notebook 02)
OBS_LTMM_CSV = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\stations data\long_term_monthly_mean\obs_ltmm_all_stations.csv"
)

# Model LTMM directory (from Notebook 01)
MODEL_LTMM_DIR = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\long_term_monthly_mean"
)

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_ROOT = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\validation\single.model"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ── Models ────────────────────────────────────────────────────────────────────
MODELS = [
    "CMCC-CM2-SR5",
    "CNRM-ESM2-1",
    "EC-Earth3-Veg",
    "IPSL-CM6A-LR",
    "MPI-ESM1-2-LR",
    "NorESM2-MM",
]

# ── Period ────────────────────────────────────────────────────────────────────
PERIOD_START = 1995
PERIOD_END   = 2014

print("Configuration loaded.")
print(f"  Models   : {len(MODELS)}")
print(f"  Period   : {PERIOD_START}–{PERIOD_END}")
print(f"  Output   : {OUTPUT_ROOT}")


Configuration loaded.
  Models   : 6
  Period   : 1995–2014
  Output   : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\single.model


## 3. Statistical Performance Metrics

Five metrics are used to comprehensively evaluate model performance. Each metric captures a different dimension of model skill:

| Metric | Formula | Optimal | Measures |
|--------|---------|---------|---------|
| **r** (Pearson's correlation) | $\frac{\sum(O-\bar{O})(M-\bar{M})}{\sqrt{\sum(O-\bar{O})^2 \sum(M-\bar{M})^2}}$ | 1.0 | Linear relationship and temporal variability |
| **NSE** (Nash-Sutcliffe Efficiency) | $1 - \frac{\sum(O-M)^2}{\sum(O-\bar{O})^2}$ | 1.0 | Predictive skill relative to observed mean |
| **PBIAS** (Percent Bias) | $\frac{\sum(M-O)}{\sum O} \times 100$ | 0.0 | Systematic over/underestimation |
| **RMSE** (Root Mean Square Error) | $\sqrt{\frac{1}{n}\sum(M-O)^2}$ | 0.0 | Average error magnitude (outlier-sensitive) |
| **MAE** (Mean Absolute Error) | $\frac{1}{n}\sum|M-O|$ | 0.0 | Average absolute error (robust) |

where $O$ = observed, $M$ = modelled, $\bar{O}$ = mean observed, $\bar{M}$ = mean modelled, $n$ = number of data points.


In [3]:
def compute_metrics(obs: np.ndarray, mod: np.ndarray) -> dict:
    """
    Compute the five evaluation metrics between observed and modelled arrays.
    Both arrays must be pre-aligned (same length, no NaN rows).

    Parameters
    ----------
    obs : array of observed values
    mod : array of modelled values

    Returns
    -------
    dict with keys: r, NSE, PBIAS, RMSE, MAE
    """
    obs = np.asarray(obs, dtype=float)
    mod = np.asarray(mod, dtype=float)

    # Drop any remaining NaN pairs
    mask = np.isfinite(obs) & np.isfinite(mod)
    obs, mod = obs[mask], mod[mask]
    n = len(obs)

    if n < 3:
        return {"r": np.nan, "NSE": np.nan, "PBIAS": np.nan,
                "RMSE": np.nan, "MAE": np.nan, "N": n}

    obs_mean = np.mean(obs)
    mod_mean = np.mean(mod)

    # Pearson's r
    num   = np.sum((obs - obs_mean) * (mod - mod_mean))
    denom = np.sqrt(np.sum((obs - obs_mean)**2) * np.sum((mod - mod_mean)**2))
    r     = num / denom if denom > 0 else np.nan

    # NSE
    ss_res = np.sum((obs - mod)**2)
    ss_tot = np.sum((obs - obs_mean)**2)
    nse    = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan

    # PBIAS (%)
    pbias  = (np.sum(mod - obs) / np.sum(obs)) * 100 if np.sum(obs) != 0 else np.nan

    # RMSE
    rmse   = np.sqrt(np.mean((mod - obs)**2))

    # MAE
    mae    = np.mean(np.abs(mod - obs))

    return {"r": round(r, 6), "NSE": round(nse, 6),
            "PBIAS": round(pbias, 4), "RMSE": round(rmse, 4),
            "MAE": round(mae, 4), "N": n}


print("Metric functions defined.")
print("  Metrics: r  |  NSE  |  PBIAS  |  RMSE  |  MAE")


Metric functions defined.
  Metrics: r  |  NSE  |  PBIAS  |  RMSE  |  MAE


## 4. Composite Ranking System

Each model is ranked on each metric individually. Rankings are then averaged to give a composite rank score (Equation 5, Hussien et al.):

$$\text{AverageRank} = \frac{\text{Rank}_r + \text{Rank}_{\text{NSE}} + \text{Rank}_{|\text{PBIAS}|} + \text{Rank}_{\text{RMSE}} + \text{Rank}_{\text{MAE}}}{5}$$

- $r$ and $\text{NSE}$: ranked **descending** (higher = better → rank 1)  
- $|\text{PBIAS}|$, $\text{RMSE}$, $\text{MAE}$: ranked **ascending** (lower = better → rank 1)  

The model with the **lowest average rank** is selected as best-performing for each basin.


In [4]:
def rank_models(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply composite ranking to a DataFrame with columns:
    [Model, r, NSE, PBIAS, RMSE, MAE]

    Returns the same DataFrame with additional columns:
    Rank_r, Rank_NSE, Rank_absPBIAS, Rank_RMSE, Rank_MAE, Avg_Rank
    Plus a 'Best_Model' boolean column.
    """
    df = metrics_df.copy()
    df["abs_PBIAS"] = df["PBIAS"].abs()

    # Rank per metric (method='average' handles ties)
    df["Rank_r"]        = df["r"].rank(ascending=False, method="average")
    df["Rank_NSE"]      = df["NSE"].rank(ascending=False, method="average")
    df["Rank_absPBIAS"] = df["abs_PBIAS"].rank(ascending=True,  method="average")
    df["Rank_RMSE"]     = df["RMSE"].rank(ascending=True,  method="average")
    df["Rank_MAE"]      = df["MAE"].rank(ascending=True,  method="average")

    df["Avg_Rank"] = (
        df["Rank_r"] + df["Rank_NSE"] + df["Rank_absPBIAS"] +
        df["Rank_RMSE"] + df["Rank_MAE"]
    ) / 5.0
    df["Avg_Rank"] = df["Avg_Rank"].round(4)

    df.drop(columns=["abs_PBIAS"], inplace=True)
    return df


print("Ranking function defined.")


Ranking function defined.


## 5. Load Observed and Modelled LTMM Data

The long-term monthly means (LTMM) computed in Notebooks 01 and 02 are loaded here. Evaluation is performed at the **basin scale**: station-level metrics are computed first, then averaged across all stations within each basin to obtain a basin-representative metric value.


In [5]:
# ── Observed LTMM ─────────────────────────────────────────────────────────────
obs_ltmm = pd.read_csv(OBS_LTMM_CSV)
obs_ltmm["Station_ID"] = obs_ltmm["Station_ID"].astype(str).str.strip()
print(f"Observed LTMM loaded : {len(obs_ltmm):,} rows  "
      f"({obs_ltmm['Station_ID'].nunique()} stations × 12 months)")

# ── Model LTMM ────────────────────────────────────────────────────────────────
model_ltmm = {}
for model in MODELS:
    csv_path = MODEL_LTMM_DIR / model / "ltmm_pr_all_stations.csv"
    if not csv_path.exists():
        print(f"  [MISSING] {model} LTMM file: {csv_path}")
        continue
    df = pd.read_csv(csv_path)
    df["Station_ID"] = df["Station_ID"].astype(str).str.strip()
    model_ltmm[model] = df
    print(f"  Loaded {model:20s} : {len(df):,} rows")

print()
print(f"Models with data : {list(model_ltmm.keys())}")

# ── Station mapping for basin assignment ──────────────────────────────────────
stations = pd.read_excel(STATION_MAPPING_FILE)
stations["Station_ID"] = stations["Station_ID"].astype(str).str.strip()
stations["Basin"]      = stations["Basin"].astype(str).str.strip()
basin_order = list(dict.fromkeys(stations["Basin"].tolist()))
print(f"\nBasins (in mapping order) : {basin_order}")


Observed LTMM loaded : 588 rows  (49 stations × 12 months)
  Loaded CMCC-CM2-SR5         : 588 rows
  Loaded CNRM-ESM2-1          : 588 rows
  Loaded EC-Earth3-Veg        : 588 rows
  Loaded IPSL-CM6A-LR         : 588 rows
  Loaded MPI-ESM1-2-LR        : 588 rows
  Loaded NorESM2-MM           : 588 rows

Models with data : ['CMCC-CM2-SR5', 'CNRM-ESM2-1', 'EC-Earth3-Veg', 'IPSL-CM6A-LR', 'MPI-ESM1-2-LR', 'NorESM2-MM']

Basins (in mapping order) : ['N.R.S.W', 'YARMOUK (JORDAN)', 'JORDAN VALLY (JORDAN)', 'AMMAN ZARQA (JORDAN)', 'S.R.S.W', 'D.S.R.S.W', 'MUJIB', 'W. ARABA NORTH', 'HASA', 'AZRAQ (JORDAN)', 'JAFER', 'HAMMAD']


## 6. Compute Station-Level Performance Metrics

For each model and each station, the five metrics are computed by comparing the 12-month observed LTMM vector against the 12-month modelled LTMM vector. This gives $6 \text{ models} \times 49 \text{ stations} = 294$ metric sets.


In [6]:
station_metric_rows = []

# Get unique station IDs that appear in observed data
obs_stations = obs_ltmm["Station_ID"].unique()

for model, mdf in model_ltmm.items():
    for sid in obs_stations:
        # Observed 12-month vector
        obs_stn = (obs_ltmm[obs_ltmm["Station_ID"] == sid]
                   .sort_values("Month")[["Month","LTMM_mm_month"]]
                   .set_index("Month"))

        # Modelled 12-month vector
        mod_stn = (mdf[mdf["Station_ID"] == sid]
                   .sort_values("Month")[["Month","LTMM_mm_month"]]
                   .set_index("Month"))

        # Align on month index
        merged = obs_stn.join(mod_stn, how="inner",
                              lsuffix="_obs", rsuffix="_mod")
        if merged.empty or len(merged) < 3:
            continue

        obs_vals = merged["LTMM_mm_month_obs"].values
        mod_vals = merged["LTMM_mm_month_mod"].values

        metrics = compute_metrics(obs_vals, mod_vals)

        # Get basin for this station
        basin_match = stations.loc[stations["Station_ID"] == sid, "Basin"]
        basin = basin_match.iloc[0] if len(basin_match) else "UNKNOWN"
        sname_match = obs_ltmm.loc[obs_ltmm["Station_ID"] == sid,
                                    "Station_Name"]
        sname = sname_match.iloc[0] if len(sname_match) else sid

        station_metric_rows.append({
            "Model"        : model,
            "Station_ID"   : sid,
            "Station_Name" : sname,
            "Basin"        : basin,
            **metrics
        })

station_metrics_df = pd.DataFrame(station_metric_rows)
station_metrics_df.to_csv(OUTPUT_ROOT / "station_level_metrics.csv", index=False)

print(f"Station-level metrics computed : {len(station_metrics_df):,} rows")
print(f"Saved: station_level_metrics.csv")
print()
print("Summary (mean across all stations per model):")
summary = (station_metrics_df.groupby("Model")[["r","NSE","RMSE","MAE","PBIAS"]]
           .mean().round(4))
print(summary.to_string())


Station-level metrics computed : 294 rows
Saved: station_level_metrics.csv

Summary (mean across all stations per model):
                    r     NSE     RMSE     MAE    PBIAS
Model                                                  
CMCC-CM2-SR5   0.9558  0.4104  10.1434  6.2453  25.8656
CNRM-ESM2-1    0.9350  0.2483  10.4952  6.5194  30.8180
EC-Earth3-Veg  0.9167  0.1971  12.2097  7.3063  30.1387
IPSL-CM6A-LR   0.9295  0.0831  11.1042  7.0443  36.3044
MPI-ESM1-2-LR  0.9680  0.5196   9.2060  5.7263  18.8568
NorESM2-MM     0.9660  0.5503   9.3729  5.9486  22.6237


## 7. Aggregate to Basin-Level Metrics

Station-level metrics are averaged across all stations within each basin to produce basin-representative performance scores. For basins with only one station, the station metric is used directly.

This follows the basin-scale analysis framework described in Section 2.5 of the manuscript, which acknowledges that precipitation characteristics vary substantially across Jordan's diverse topography.


In [7]:
basin_metric_rows = []

for model in MODELS:
    if model not in model_ltmm:
        continue
    model_stn = station_metrics_df[station_metrics_df["Model"] == model]

    for basin in basin_order:
        basin_stn = model_stn[model_stn["Basin"] == basin]

        if basin_stn.empty:
            basin_metric_rows.append({
                "Model": model, "Basin": basin,
                "r": np.nan, "NSE": np.nan, "PBIAS": np.nan,
                "RMSE": np.nan, "MAE": np.nan,
                "N_Stations": 0
            })
            continue

        basin_metric_rows.append({
            "Model"      : model,
            "Basin"      : basin,
            "r"          : round(basin_stn["r"].mean(),    6),
            "NSE"        : round(basin_stn["NSE"].mean(),  6),
            "PBIAS"      : round(basin_stn["PBIAS"].mean(),4),
            "RMSE"       : round(basin_stn["RMSE"].mean(), 4),
            "MAE"        : round(basin_stn["MAE"].mean(),  4),
            "N_Stations" : len(basin_stn),
        })

basin_metrics_df = pd.DataFrame(basin_metric_rows)
basin_metrics_df.to_csv(OUTPUT_ROOT / "basin_level_metrics.csv", index=False)

print(f"Basin-level metrics computed : {len(basin_metrics_df):,} rows")
print(f"Saved: basin_level_metrics.csv")
print()
print("Basin metrics table (all models):")
disp = basin_metrics_df[["Model","Basin","r","NSE","PBIAS","RMSE","MAE"]]
print(disp.to_string(index=False))


Basin-level metrics computed : 72 rows
Saved: basin_level_metrics.csv

Basin metrics table (all models):
        Model                 Basin        r       NSE    PBIAS    RMSE     MAE
 CMCC-CM2-SR5               N.R.S.W 0.975787  0.884669  -7.5578 14.2064  8.2960
 CMCC-CM2-SR5      YARMOUK (JORDAN) 0.958603  0.744681  27.1671 11.0171  6.8584
 CMCC-CM2-SR5 JORDAN VALLY (JORDAN) 0.986484  0.575480  46.0821 16.2658 10.8046
 CMCC-CM2-SR5  AMMAN ZARQA (JORDAN) 0.976724  0.357811  37.6110 11.7388  7.6232
 CMCC-CM2-SR5               S.R.S.W 0.984051  0.701358  -1.0436 15.8240  9.6631
 CMCC-CM2-SR5             D.S.R.S.W 0.959564  0.883577  -8.9586  8.6978  4.9947
 CMCC-CM2-SR5                 MUJIB 0.955541  0.424232  31.5972  8.6313  5.4636
 CMCC-CM2-SR5        W. ARABA NORTH 0.932719  0.785447 -17.8017  9.6095  5.1694
 CMCC-CM2-SR5                  HASA 0.948647  0.878169  -6.6041  5.9342  3.4832
 CMCC-CM2-SR5        AZRAQ (JORDAN) 0.922860 -1.375626 114.6291  6.9148  4.7072
 CMCC-CM2-SR5  

## 8. Composite Ranking — Best Model per Basin

For each basin, models are ranked on each of the five metrics and the composite average rank is computed. The model with the lowest average rank is selected as the best-performing model for that basin.


In [8]:
ranking_rows = []
best_model_per_basin = {}

for basin in basin_order:
    basin_df = basin_metrics_df[basin_metrics_df["Basin"] == basin].copy()

    if basin_df["r"].isna().all():
        print(f"  [SKIP] {basin} — no metric data.")
        continue

    ranked = rank_models(basin_df[["Model","r","NSE","PBIAS","RMSE","MAE"]])
    ranked["Basin"] = basin
    ranked = ranked.sort_values("Avg_Rank").reset_index(drop=True)
    ranked["Basin_Rank"] = ranked.index + 1

    # Merge N_Stations back
    ranked = ranked.merge(
        basin_metrics_df[["Model","Basin","N_Stations"]],
        on=["Model","Basin"], how="left"
    )

    ranking_rows.append(ranked)

    best = ranked.iloc[0]
    best_model_per_basin[basin] = {
        "Best_Model": best["Model"],
        "Avg_Rank"  : best["Avg_Rank"],
        "r"         : best["r"],
        "NSE"       : best["NSE"],
        "PBIAS"     : best["PBIAS"],
        "RMSE"      : best["RMSE"],
        "MAE"       : best["MAE"],
        "N_Stations": int(best["N_Stations"]) if not np.isnan(best["N_Stations"]) else 0,
    }

ranking_df = pd.concat(ranking_rows, ignore_index=True)
ranking_df.to_csv(OUTPUT_ROOT / "basin_model_rankings.csv", index=False)

print("Composite ranking complete.")
print(f"Saved: basin_model_rankings.csv")
print()
print("=" * 72)
print("BEST-PERFORMING MODEL PER BASIN  (Table 3 equivalent)")
print("=" * 72)
print(f"{'Basin':<25} {'Best Model':<22} {'r':>6} {'NSE':>7} {'PBIAS':>7} {'RMSE':>7} {'MAE':>7}")
print("-" * 72)
for basin, info in best_model_per_basin.items():
    print(f"{basin:<25} {info['Best_Model']:<22} "
          f"{info['r']:>6.3f} {info['NSE']:>7.3f} "
          f"{info['PBIAS']:>7.2f} {info['RMSE']:>7.3f} {info['MAE']:>7.3f}")


Composite ranking complete.
Saved: basin_model_rankings.csv

BEST-PERFORMING MODEL PER BASIN  (Table 3 equivalent)
Basin                     Best Model                  r     NSE   PBIAS    RMSE     MAE
------------------------------------------------------------------------
N.R.S.W                   NorESM2-MM              0.984   0.902   -6.77  12.792   7.612
YARMOUK (JORDAN)          MPI-ESM1-2-LR           0.965   0.808   20.19   8.960   5.439
JORDAN VALLY (JORDAN)     MPI-ESM1-2-LR           0.987   0.649   40.42  14.736   9.827
AMMAN ZARQA (JORDAN)      MPI-ESM1-2-LR           0.984   0.494   29.33  10.831   6.965
S.R.S.W                   MPI-ESM1-2-LR           0.995   0.760   -6.50  14.757   9.286
D.S.R.S.W                 IPSL-CM6A-LR            0.967   0.905   -3.75   8.051   4.973
MUJIB                     MPI-ESM1-2-LR           0.973   0.590   22.05   7.489   4.802
W. ARABA NORTH            IPSL-CM6A-LR            0.916   0.793   -9.25   9.359   5.372
HASA                

## 9. Best-Model Summary Table

A clean summary table  showing the best-performing model for each basin with its five metric values. This table is saved as both CSV and a formatted Excel file.


In [9]:
# Build Table 3 equivalent
table3_rows = []
for basin, info in best_model_per_basin.items():
    table3_rows.append({
        "Basin"      : basin,
        "Best_Model" : info["Best_Model"],
        "r"          : info["r"],
        "NSE"        : info["NSE"],
        "PBIAS"      : info["PBIAS"],
        "RMSE"       : info["RMSE"],
        "MAE"        : info["MAE"],
    })

table3_df = pd.DataFrame(table3_rows)

# Add mean and std rows
numeric_cols = ["r","NSE","PBIAS","RMSE","MAE"]
mean_row = {"Basin": "Mean", "Best_Model": "—"}
std_row  = {"Basin": "Std Dev", "Best_Model": "—"}
for col in numeric_cols:
    mean_row[col] = round(table3_df[col].mean(), 3)
    std_row[col]  = round(table3_df[col].std(),  3)

table3_df = pd.concat(
    [table3_df, pd.DataFrame([mean_row, std_row])],
    ignore_index=True
)

table3_df.to_csv(OUTPUT_ROOT / "table3_best_model_per_basin.csv", index=False)

# ── Excel version with formatting ─────────────────────────────────────────────
try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    ws = wb.active
    ws.title = "Table 3 - Best Model"

    thin   = Side(style="thin", color="AAAAAA")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    center = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left   = Alignment(horizontal="left",   vertical="center")
    hdr_fill  = PatternFill("solid", start_color="1F4E79")
    mean_fill = PatternFill("solid", start_color="EBF5FB")
    std_fill  = PatternFill("solid", start_color="F8F9FA")
    title_font = Font(name="Arial", bold=True, size=12, color="1F4E79")

    # Title
    ws.merge_cells("A1:G1")
    ws["A1"].value     = "Table 3. Model Performance Evaluation Results Across Jordan's Hydrological Basins (1995–2014)"
    ws["A1"].font      = title_font
    ws["A1"].alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 22

    # Headers
    headers = ["Basin", "Best Model", "Correlation (r)", "NSE", "PBIAS (%)", "RMSE (mm/month)", "MAE (mm/month)"]




    col_widths = [22, 20, 13, 10, 10, 14, 14]
    for ci, (h, w) in enumerate(zip(headers, col_widths), 1):
        c = ws.cell(row=2, column=ci, value=h)
        c.fill      = hdr_fill
        c.font      = Font(name="Arial", bold=True, color="FFFFFF", size=9)
        c.alignment = center
        c.border    = border
        ws.column_dimensions[get_column_letter(ci)].width = w
    ws.row_dimensions[2].height = 32

    # Data rows
    for ri, row in table3_df.iterrows():
        r = ri + 3
        is_stat = row["Basin"] in ("Mean","Std Dev")
        row_fill = mean_fill if row["Basin"] == "Mean" else (
                   std_fill  if row["Basin"] == "Std Dev" else None)
        vals = [row["Basin"], row["Best_Model"],
                row["r"], row["NSE"], row["PBIAS"],
                row["RMSE"], row["MAE"]]
        for ci, v in enumerate(vals, 1):
            cell = ws.cell(row=r, column=ci)
            cell.value  = v if v not in [np.nan, "—"] else ("—" if isinstance(v,str) else "")
            cell.font   = Font(name="Arial", bold=is_stat, size=9)
            cell.alignment = left if ci <= 2 else center
            cell.border = border
            if row_fill:
                cell.fill = row_fill
            if ci > 2 and isinstance(v, float) and not np.isnan(v):
                cell.number_format = "0.000"

    ws.freeze_panes = "A3"
    excel_out = OUTPUT_ROOT / "table3_best_model_per_basin.xlsx"
    wb.save(excel_out)
    print(f"Excel table saved: {excel_out}")
except Exception as e:
    print(f"Excel formatting skipped: {e}")

print()
print("Table 3 preview:")
print(table3_df.to_string(index=False))


Excel table saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\single.model\table3_best_model_per_basin.xlsx

Table 3 preview:
                Basin    Best_Model        r       NSE    PBIAS    RMSE    MAE
              N.R.S.W    NorESM2-MM 0.983814  0.902149  -6.7718 12.7923 7.6117
     YARMOUK (JORDAN) MPI-ESM1-2-LR 0.964858  0.808411  20.1873  8.9605 5.4393
JORDAN VALLY (JORDAN) MPI-ESM1-2-LR 0.987190  0.648509  40.4249 14.7365 9.8267
 AMMAN ZARQA (JORDAN) MPI-ESM1-2-LR 0.984215  0.493637  29.3291 10.8312 6.9649
              S.R.S.W MPI-ESM1-2-LR 0.995172  0.759890  -6.4953 14.7574 9.2864
            D.S.R.S.W  IPSL-CM6A-LR 0.967026  0.905413  -3.7507  8.0511 4.9726
                MUJIB MPI-ESM1-2-LR 0.973303  0.590338  22.0534  7.4894 4.8021
       W. ARABA NORTH  IPSL-CM6A-LR 0.916434  0.792734  -9.2451  9.3593 5.3718
                 HASA MPI-ESM1-2-LR 0.978433  0.919758 -11.5780  4.8159 2.8414
       AZRAQ (JORDAN) MPI-ESM1-2-LR 0.910454 -0.923240 102.3180

## 10. Model Selection Frequency

How often each model is selected as the best-performing model across all basins


In [10]:
freq = {}
for model in MODELS:
    freq[model] = sum(1 for info in best_model_per_basin.values()
                      if info["Best_Model"] == model)

freq_df = pd.DataFrame([
    {"Model": m, "N_Basins_Best": freq.get(m, 0)}
    for m in MODELS
]).sort_values("N_Basins_Best", ascending=False)

print("Model selection frequency (best-performing model per basin):")
print()
for _, row in freq_df.iterrows():
    bars = "█" * int(row["N_Basins_Best"])
    print(f"  {row['Model']:<22} {bars:<14} {row['N_Basins_Best']} basins")

freq_df.to_csv(OUTPUT_ROOT / "model_selection_frequency.csv", index=False)
print()
print(f"Saved: model_selection_frequency.csv")


Model selection frequency (best-performing model per basin):

  MPI-ESM1-2-LR          ████████       8 basins
  IPSL-CM6A-LR           ██             2 basins
  NorESM2-MM             ██             2 basins
  CMCC-CM2-SR5                          0 basins
  CNRM-ESM2-1                           0 basins
  EC-Earth3-Veg                         0 basins

Saved: model_selection_frequency.csv


## 11. Ensemble Approach Comparison

Three modeling approaches are compared at basin level 


1. **Best single model** — the top-ranked model per basin  
2. **3-model ensemble** — arithmetic mean of the three top-ranked models per basin  
3. **6-model ensemble** — arithmetic mean of all six models  

For each approach, the basin-level LTMM is constructed from the model LTMM CSVs and metrics are re-computed against observations.


In [11]:
# Build a wide LTMM table: Station_ID × Month → one column per model
ltmm_wide = obs_ltmm[["Station_ID","Station_Name","Basin","Month","LTMM_mm_month"]].copy()
ltmm_wide = ltmm_wide.rename(columns={"LTMM_mm_month": "Obs"})

for model, mdf in model_ltmm.items():
    mdf_sub = (mdf[["Station_ID","Month","LTMM_mm_month"]]
               .rename(columns={"LTMM_mm_month": model}))
    ltmm_wide = ltmm_wide.merge(mdf_sub, on=["Station_ID","Month"], how="left")

ltmm_wide.to_csv(OUTPUT_ROOT / "ltmm_wide_obs_and_models.csv", index=False)
print(f"Wide LTMM table: {ltmm_wide.shape}  → ltmm_wide_obs_and_models.csv")

# ── Compute basin metrics for each approach ───────────────────────────────────
approach_results = []

for basin in basin_order:
    basin_data = ltmm_wide[ltmm_wide["Basin"] == basin].copy()
    if basin_data.empty:
        continue

    # Get top-3 models for this basin from ranking
    basin_rank = ranking_df[ranking_df["Basin"] == basin].sort_values("Avg_Rank")
    top3 = basin_rank["Model"].iloc[:3].tolist() if len(basin_rank) >= 3 else basin_rank["Model"].tolist()
    top1 = basin_rank["Model"].iloc[0] if len(basin_rank) >= 1 else None

    obs_vals = basin_data["Obs"].values

    # Approach 1: best single model
    if top1 and top1 in basin_data.columns:
        mod1 = basin_data[top1].values
        m1   = compute_metrics(obs_vals, mod1)
        approach_results.append({"Approach":"Best Single Model","Basin":basin,**m1})

    # Approach 2: 3-model ensemble (mean of top-3 LTMM)
    if top3:
        top3_valid = [m for m in top3 if m in basin_data.columns]
        if top3_valid:
            ens3 = basin_data[top3_valid].mean(axis=1).values
            m3   = compute_metrics(obs_vals, ens3)
            approach_results.append({"Approach":"3-Model Ensemble","Basin":basin,**m3})

    # Approach 3: 6-model ensemble
    all_models_present = [m for m in MODELS if m in basin_data.columns]
    if all_models_present:
        ens6 = basin_data[all_models_present].mean(axis=1).values
        m6   = compute_metrics(obs_vals, ens6)
        approach_results.append({"Approach":"6-Model Ensemble","Basin":basin,**m6})

approach_df = pd.DataFrame(approach_results)
approach_df.to_csv(OUTPUT_ROOT / "ensemble_comparison_basin.csv", index=False)
print(f"\nEnsemble comparison saved: ensemble_comparison_basin.csv")

# ── Summary statistics across basins ─────────────────────────────────────────
print()
print("=" * 72)
print("ENSEMBLE APPROACH COMPARISON — BASIN-LEVEL MEDIANS")
print("=" * 72)
summary_ens = (approach_df.groupby("Approach")[["r","NSE","RMSE","MAE","PBIAS"]]
               .median().round(3))
# Reorder rows
order_map = {"Best Single Model": 0, "3-Model Ensemble": 1, "6-Model Ensemble": 2}
summary_ens = summary_ens.loc[
    sorted(summary_ens.index, key=lambda x: order_map.get(x, 9))
]
print(summary_ens.to_string())


Wide LTMM table: (588, 11)  → ltmm_wide_obs_and_models.csv

Ensemble comparison saved: ensemble_comparison_basin.csv

ENSEMBLE APPROACH COMPARISON — BASIN-LEVEL MEDIANS
                       r    NSE   RMSE    MAE   PBIAS
Approach                                             
Best Single Model  0.932  0.818  8.808  5.172   5.370
3-Model Ensemble   0.932  0.813  8.651  5.015   9.214
6-Model Ensemble   0.919  0.790  9.367  5.193  11.987


## 12. Output Directory Summary


In [12]:
print("=" * 72)
print("VALIDATION OUTPUT FILES")
print("=" * 72)
for root, dirs, files in os.walk(OUTPUT_ROOT):
    dirs[:] = sorted(d for d in dirs if not d.startswith("."))
    level   = Path(root).relative_to(OUTPUT_ROOT).parts
    indent  = "  " * len(level)
    print(f"{indent}📁 {Path(root).name}/")
    sub = "  " * (len(level) + 1)
    for f in sorted(files):
        sz = (Path(root)/f).stat().st_size / 1024
        unit = "KB"
        if sz > 1024:
            sz /= 1024; unit = "MB"
        print(f"{sub}📄 {f}  ({sz:.1f} {unit})")

print()
print("=" * 72)
print("VALIDATION COMPLETE")
print("=" * 72)
print()
print("Output files summary:")
print("  station_level_metrics.csv         — 294 rows (49 stations × 6 models)")
print("  basin_level_metrics.csv           — basin-averaged metrics per model")
print("  basin_model_rankings.csv          — composite rankings per basin")
print("  table3_best_model_per_basin.csv   — Table 3 equivalent (CSV)")
print("  table3_best_model_per_basin.xlsx  — Table 3 equivalent (formatted Excel)")
print("  model_selection_frequency.csv     — how often each model ranked best")
print("  ltmm_wide_obs_and_models.csv      — observations + all model LTMM wide")
print("  ensemble_comparison_basin.csv     — 3-approach comparison per basin")


VALIDATION OUTPUT FILES
📁 single.model/
  📄 basin_level_metrics.csv  (4.8 KB)
  📄 basin_model_rankings.csv  (6.7 KB)
  📄 ensemble_comparison_basin.csv  (2.6 KB)
  📄 ltmm_wide_obs_and_models.csv  (47.8 KB)
  📄 model_selection_frequency.csv  (0.1 KB)
  📄 station_level_metrics.csv  (25.9 KB)
  📄 table3_best_model_per_basin.csv  (0.9 KB)
  📄 table3_best_model_per_basin.xlsx  (6.2 KB)

VALIDATION COMPLETE

Output files summary:
  station_level_metrics.csv         — 294 rows (49 stations × 6 models)
  basin_level_metrics.csv           — basin-averaged metrics per model
  basin_model_rankings.csv          — composite rankings per basin
  table3_best_model_per_basin.csv   — Table 3 equivalent (CSV)
  table3_best_model_per_basin.xlsx  — Table 3 equivalent (formatted Excel)
  model_selection_frequency.csv     — how often each model ranked best
  ltmm_wide_obs_and_models.csv      — observations + all model LTMM wide
  ensemble_comparison_basin.csv     — 3-approach comparison per basin
